# Clase 4 — Orquestadores: Apache Airflow

**Objetivo:** entender DAGs, tareas, dependencias, XCom y *schedules*, leer los DAGs del TP y
**ejecutar su lógica** paso a paso.

> Airflow no corre nativo en Windows (necesita Linux: Docker o WSL). En el TP corre dentro del
> docker-compose. Acá ejecutamos **el mismo código de las tareas** reemplazando MinIO por una carpeta
> local, para ver qué hace cada paso sin levantar nada.

In [1]:
from helpers import OUT, ROOT, check, quiz, safe_check

import ast
import io
import json
import shutil
from pathlib import Path

import pandas as pd
from ipywidgets import interact, widgets

## 1. Sincronizadores vs orquestadores
- Un **sincronizador** (cron) decide *cuándo* corre algo.
- Un **orquestador** (Airflow) además maneja *qué depende de qué*, reintentos, historial, logs y
  paralelismo.

| Concepto de Airflow | Qué es | En el TP |
|---|---|---|
| **DAG** | grafo dirigido acíclico de tareas | `etl_burnout`, `train_burnout`, `batch_predict_burnout` |
| **Tarea** | un paso del DAG | `extract`, `validate_raw`, `split`… |
| **Operador** | tipo de tarea (`PythonOperator`, `BashOperator`…) | usamos `@task` (TaskFlow) |
| **XCom** | valores chicos que pasan entre tareas | el resumen del dataset (`dict`) |
| **Scheduler** | decide qué tarea corre y cuándo | contenedor `airflow-scheduler` |
| **Executor / worker** | ejecuta las tareas | `CeleryExecutor` + `airflow-worker` (broker: Valkey) |
| **Variables / Connections** | configuración externa | `airflow/secrets/variables.yaml` |

In [2]:
quiz(
    "¿Por qué un DAG tiene que ser *acíclico*?",
    ["Por estética", "Si A depende de B y B de A, ninguna podría empezar", "Porque Airflow no soporta más de 10 tareas"],
    answer=1,
    explanation="Un ciclo es una dependencia circular: no hay orden de ejecución posible.",
)

**¿Por qué un DAG tiene que ser *acíclico*?**

ToggleButtons(options=('Por estética', 'Si A depende de B y B de A, ninguna podría empezar', 'Porque Airflow n…

Output()

## 2. TaskFlow: cómo se ve un DAG del TP
Con la API TaskFlow (Airflow 3: `from airflow.sdk import dag, task`) una tarea es una función con
`@task`, y **pasar el resultado de una función a otra define la dependencia** (y viaja por XCom).

In [3]:
print((ROOT / "airflow" / "dags" / "etl_burnout.py").read_text(encoding="utf-8")[:2600])
print("…")

"""DAG de ETL: lleva el dataset crudo al Data Lake y deja listos train y test."""

from datetime import timedelta

from airflow.sdk import dag, task

DOC_MD = """
### ETL del dataset de burnout

Toma el CSV de [Impact of AI on Students](https://www.kaggle.com/datasets/laveshjadon/ai-impact-on-students)
(versionado en `data/raw/` del repo), lo sube a `s3://data/burnout/raw/`, valida el esquema y
genera el split 80/20 estratificado (semilla 42) en `s3://data/burnout/final/`.

Al final guarda `s3://data/burnout/data_info/data.json` con el esquema y registra los datasets
en MLflow (experimento *Burnout Risk*), para que cada entrenamiento sepa con qué datos se hizo.
"""

default_args = {
    "owner": "Grupo AVS",
    "depends_on_past": False,
    "retries": 1,
    "retry_delay": timedelta(minutes=2),
}


@dag(
    dag_id="etl_burnout",
    description="ETL: CSV crudo → validación → split train/test en MinIO",
    doc_md=DOC_MD,
    schedule=None,
    catchup=False,
    dagrun_timeout=timede

### 🎛️ Explorá los tres DAGs
Leemos los archivos con `ast` (sin importar Airflow) y listamos sus tareas con su docstring.

In [4]:
def tareas_del_dag(path: Path) -> pd.DataFrame:
    arbol = ast.parse(path.read_text(encoding="utf-8"))
    filas = []
    for nodo in ast.walk(arbol):
        if isinstance(nodo, ast.FunctionDef) and any(
            (isinstance(d, ast.Name) and d.id == "task") or (isinstance(d, ast.Attribute) and d.attr == "task")
            for d in nodo.decorator_list
        ):
            filas.append({"tarea": nodo.name, "qué hace": (ast.get_docstring(nodo) or "").splitlines()[0]})
    return pd.DataFrame(filas)


dags = {p.stem: p for p in sorted((ROOT / "airflow" / "dags").glob("*.py"))}


@interact(dag=widgets.Dropdown(options=list(dags)))
def ver_dag(dag):
    texto = dags[dag].read_text(encoding="utf-8")
    llamada = [line.strip() for line in texto.splitlines() if line.strip().endswith("))") and "(" in line][-1]
    display(tareas_del_dag(dags[dag]))
    print("Dependencias (última línea del DAG):", llamada)

interactive(children=(Dropdown(description='dag', options=('batch_predict_burnout', 'etl_burnout', 'train_burn…

`register_dataset(split(validate_raw(extract())))` se lee de adentro hacia afuera:
**extract → validate_raw → split → register_dataset**. Equivale a `extract >> validate_raw >> split >> register_dataset`
del estilo tradicional.

## 3. Ejecutemos la lógica del ETL con un "S3 falso"
Nuestras tareas leen y escriben con `burnout.storage`, que usa `boto3`. Reemplazamos el cliente de
boto3 por uno que guarda en `clases/_salidas/s3/`. Es la técnica de *mocking* que también sirve
para testear.

In [5]:
import burnout.storage as storage
from botocore.exceptions import ClientError

S3_LOCAL = OUT / "s3"
shutil.rmtree(S3_LOCAL, ignore_errors=True)


class S3Falso:
    """Implementa get_object/put_object de boto3 sobre una carpeta local."""

    def put_object(self, Bucket, Key, Body):
        destino = S3_LOCAL / Bucket / Key
        destino.parent.mkdir(parents=True, exist_ok=True)
        destino.write_bytes(Body)

    def get_object(self, Bucket, Key):
        origen = S3_LOCAL / Bucket / Key
        if not origen.exists():
            raise ClientError({"Error": {"Code": "NoSuchKey"}}, "GetObject")
        return {"Body": io.BytesIO(origen.read_bytes())}


storage._client = lambda: S3Falso()  # el "mock"
print("Ahora burnout.storage escribe en", S3_LOCAL)

Ahora burnout.storage escribe en C:\Users\USUARIO\CEIA\MLOps-TP-AVS\clases\_salidas\s3


Ahora corremos, en orden, **las mismas funciones** que llaman las tareas del DAG `etl_burnout`
(están en `src/burnout/etl.py`; el DAG solo las envuelve con `@task`). Fijate cómo cada paso le pasa
al siguiente un valor chico (el XCom) y los datos grandes viajan por el Data Lake.

In [6]:
import inspect

from burnout import etl
from burnout.config import S3_KEYS
from burnout.etl import extract, split, validate_raw

print(inspect.getsource(etl.split))

def split(summary: dict) -> dict:
    """Genera el split 80/20 estratificado y guarda X/y de train y test en ``final/``."""
    df = validate(storage.read_csv(S3_KEYS["raw"]))
    X_train, X_test, y_train, y_test = make_split(df)
    frames = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train.to_frame(),
        "y_test": y_test.to_frame(),
    }
    for name, frame in frames.items():
        storage.write_csv(frame, S3_KEYS[name])
    return {**summary, "train_rows": len(X_train), "test_rows": len(X_test)}



In [7]:
xcom_1 = extract()
print("extract      → XCom:", xcom_1)
xcom_2 = validate_raw(xcom_1)
print("validate_raw → XCom:", json.dumps(xcom_2))
xcom_3 = split(xcom_2)
print("split        → XCom:", json.dumps(xcom_3))

extract      → XCom:

 s3://data/burnout/raw/ai_student_impact_dataset.csv
validate_raw → XCom: {"source": "s3://data/burnout/raw/ai_student_impact_dataset.csv", "rows": 50000, "class_share": {"Medium": 0.4229, "Low": 0.3274, "High": 0.2497}}


split        → XCom: {"source": "s3://data/burnout/raw/ai_student_impact_dataset.csv", "rows": 50000, "class_share": {"Medium": 0.4229, "Low": 0.3274, "High": 0.2497}, "train_rows": 40000, "test_rows": 10000}


In [8]:
print("Archivos en el Data Lake simulado:")
for f in sorted(S3_LOCAL.rglob("*.csv")):
    print(f"  s3://{f.relative_to(S3_LOCAL).as_posix():<55} {f.stat().st_size / 1e6:6.2f} MB")

Archivos en el Data Lake simulado:
  s3://data/burnout/final/test/X_test.csv                        0.91 MB
  s3://data/burnout/final/test/y_test.csv                        0.03 MB
  s3://data/burnout/final/train/X_train.csv                      3.65 MB
  s3://data/burnout/final/train/y_train.csv                      0.12 MB
  s3://data/burnout/raw/ai_student_impact_dataset.csv            5.94 MB


### ¿Qué pasa si los datos vienen mal?
Rompamos el CSV crudo en el "S3" y volvamos a correr `validate_raw`. En Airflow, la excepción marca
la tarea como **failed**, se reintenta (`retries=1`) y las siguientes **no se ejecutan**: datos
malos nunca llegan a entrenar un modelo.

In [9]:
crudo = storage.read_csv(S3_KEYS["raw"])
crudo.loc[0, "Institutional_Policy"] = "Strictly_Ban"  # el valor que decía Kaggle, no el del CSV
storage.write_csv(crudo, S3_KEYS["raw"])
try:
    validate_raw(xcom_1)
except ValueError as e:
    print("❌ La tarea validate_raw fallaría con:", e)
extract()  # restauramos el crudo

❌ La tarea validate_raw fallaría con: Categorías desconocidas en Institutional_Policy: ['Strictly_Ban']


's3://data/burnout/raw/ai_student_impact_dataset.csv'

### ✏️ Ejercicio 1 — orden de ejecución
El scheduler calcula el orden con un **orden topológico** del grafo. Completá `orden` usando
`graphlib.TopologicalSorter` de la librería estándar para este DAG de la segunda entrega (cada
tarea → conjunto de tareas de las que depende).

In [10]:
from graphlib import TopologicalSorter

dependencias = {
    "evaluar_champion_vs_challenger": {"entrenar_challenger", "cargar_test"},
    "entrenar_challenger": {"cargar_train"},
    "cargar_train": {"etl"},
    "cargar_test": {"etl"},
    "etl": set(),
}
orden = None  # ← tu código acá (una lista)

In [11]:
def _ej1():
    pos = {t: i for i, t in enumerate(orden)}
    return set(orden) == set(dependencias) and all(pos[d] < pos[t] for t, ds in dependencias.items() for d in ds)


safe_check("Ejercicio 1", _ej1, "`orden = list(TopologicalSorter(dependencias).static_order())`.")

⏳ **Ejercicio 1**: todavía no. *Pista:* `orden = list(TopologicalSorter(dependencias).static_order())`. (error: `TypeError: 'NoneType' object is not iterable`)

False

<details><summary>Ver solución</summary>

```python
orden = list(TopologicalSorter(dependencias).static_order())
```
`cargar_train` y `cargar_test` no dependen entre sí: Airflow las puede correr **en paralelo**.
</details>

## 4. Schedules (expresiones cron)
`minuto hora día_del_mes mes día_de_la_semana`. También hay atajos: `@daily`, `@hourly`, `@weekly`.
`batch_predict_burnout` usa `@daily`; `etl_burnout` y `train_burnout` usan `schedule=None` (se
disparan a mano o desde otro DAG).

In [12]:
quiz(
    "¿Qué significa schedule=\"0 3 * * 1\"?",
    ["Cada 3 minutos", "Los lunes a las 03:00", "El día 3 de cada mes"],
    answer=1,
    explanation="minuto 0, hora 3, cualquier día del mes, cualquier mes, día de la semana 1 (lunes).",
)

**¿Qué significa schedule="0 3 * * 1"?**

ToggleButtons(options=('Cada 3 minutos', 'Los lunes a las 03:00', 'El día 3 de cada mes'), style=ToggleButtons…

Output()

### ✏️ Ejercicio 2
Escribí la expresión cron para "todos los días a las 06:30".

In [13]:
mi_cron = ""  # ← tu respuesta

In [14]:
check("Ejercicio 2", mi_cron.split() == ["30", "6", "*", "*", "*"], "el minuto va primero: `30 6 * * *`.")

⏳ **Ejercicio 2**: todavía no. *Pista:* el minuto va primero: `30 6 * * *`.

False

## 5. Detalles de Airflow 3 que aparecen en el material
- Imports nuevos: `from airflow.sdk import dag, task, Variable` (los de `airflow.decorators` están deprecados).
- `schedule_interval` **ya no existe**: se usa `schedule=`. Ponerlo en `default_args` (como algunos
  ejemplos de la cátedra) no tiene efecto.
- `catchup=False` evita que Airflow corra todas las ejecuciones "atrasadas" desde `start_date`.
- `@task.virtualenv` crea un entorno por tarea (aislado pero lento: instala paquetes en cada corrida).
  En el TP preferimos instalar las librerías **en la imagen** y usar `@task` común.
- Los imports pesados (pandas, mlflow) van **dentro** de las tareas: el *dag-processor* lee los
  archivos de DAG cada pocos segundos y así no carga esas librerías cada vez.

In [15]:
quiz(
    "¿Por qué los DAGs del TP importan pandas y mlflow DENTRO de cada @task?",
    ["Es obligatorio en Airflow 3", "Para que el parseo periódico del archivo de DAG sea rápido", "Para ahorrar disco"],
    answer=1,
    explanation="El dag-processor importa el archivo muy seguido; con imports dentro de las tareas, solo los "
    "carga el worker cuando la tarea corre.",
)

**¿Por qué los DAGs del TP importan pandas y mlflow DENTRO de cada @task?**

ToggleButtons(options=('Es obligatorio en Airflow 3', 'Para que el parseo periódico del archivo de DAG sea ráp…

Output()

## 6. Correrlo de verdad (con Docker)
1. `docker compose --profile all up -d` y entrar a http://localhost:8080 (usuario y clave: `airflow`).
2. Activar y disparar (**Trigger**) en este orden: `etl_burnout` → `train_burnout` → `batch_predict_burnout`.
3. Ver los logs de cada tarea en la vista **Grid**, y los resultados en MinIO (http://localhost:9001,
   bucket `data/burnout/`) y en MLflow (http://localhost:5001).

Por línea de comandos:

```bash
docker compose run --rm airflow-cli dags list
docker compose run --rm airflow-cli dags trigger etl_burnout
```

## Resumen
- DAG = tareas + dependencias, sin ciclos; el scheduler las ordena topológicamente.
- TaskFlow: pasar resultados entre `@task` define dependencias (XCom); los datos grandes van al Data Lake.
- Una tarea que falla corta el flujo: la validación protege al entrenamiento.

Siguiente: **C5 — Despliegue y predicción por lotes**.